# Automated Test Runner — Single Course

Creates a **single multi-task Lakeflow Job** for one course, with optional QA checks,
so the full run is visible in one timeline.

**Run structure:**
- Course notebooks run in the dependency order you define in `COURSE_TASKS`.
- **QA Content Checker** can run independently (no dependencies) alongside the course tasks.
- The separate **Course-specific inputs** cell makes it easy to reuse this notebook for another course.

```
 QA Check      :  qa_content_checker  (optional, parallel)
 Course Tasks  :  task_01  ──►  task_02  ──►  task_03
```

**What it does**
1. Reads course-specific settings from a dedicated configuration cell (`COURSE_NAME`, `LAB_NOTEBOOKS`, `COURSE_TASKS`).
2. Auto-fills all `<FILL_IN>` placeholders in the configured lab notebooks using the inline solution blocks.
3. Resolves all notebook paths relative to this notebook's location.
4. Creates (or re-creates) a single persistent Lakeflow Job for the configured course.
5. Optionally adds `qa_content_checker` as an **independent task** (no dependencies).
6. Triggers the job with `jobs.run_now` and polls until completion.
7. Appends one result row per task to a generic results table.
8. Displays a combined summary table with clickable `run_page_url` links.
9. Creates and publishes a **Lakeview dashboard** from the run results and QA findings.
10. Raises if any task failed — triggering Lakeflow Job failure email.

In [0]:
%run ./notebook_path

## Notebooks with UI Instructions (Auto-Scanned)

The following cell **dynamically scans** all course notebooks (demos and labs) to detect manual UI steps that cannot be fully automated by the test runner.

It looks for markdown cells containing UI action patterns (e.g., "Select...", "Click...", "Navigate to...", icon references, etc.) and reports which notebooks have them.

**This runs fresh on every execution** — no hardcoded values — so it always reflects the latest course content.

In [0]:
# ── Auto-Scan: Notebooks with UI Instructions ───────────────────────────────
# Dynamically scans ALL course notebooks (demos and labs) to detect cells
# containing manual UI steps that cannot be automated by the test runner.
#
# NOT HARDCODED — runs fresh every time so it reflects the latest course content.
# ─────────────────────────────────────────────────────────────────────────────

import base64
import json
import re
import os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat

w = WorkspaceClient()

# Derive course_root from this notebook's own path
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
course_root = "/".join(_nb_path.split("/")[:-1])

# ── UI Detection Patterns ─────────────────────────────────────────────────────
# These regex patterns identify markdown cells that contain UI interaction steps.
# They match common instructional language used in Databricks Academy notebooks.

UI_ACTION_PATTERNS = [
    # Direct UI actions
    re.compile(r'\bSelect\s+(the\s+)?\*\*', re.IGNORECASE),          # "Select the **Catalog** icon"
    re.compile(r'\bClick\s+(the\s+|on\s+)?\*\*', re.IGNORECASE),    # "Click the **Run pipeline** button"
    re.compile(r'\bright-click\b', re.IGNORECASE),                    # "right-click on Jobs & Pipelines"
    re.compile(r'\bOpen\s+in\s+(a\s+)?(New|new)\s+Tab\b'),           # "Open in New Tab"
    re.compile(r'\bOpen\s+(Link\s+)?in\s+New\s+(Browser\s+)?Tab\b', re.IGNORECASE),
    # Navigation instructions
    re.compile(r'\b(left|top|main|far-left)\s+navigation\s+(bar|pane)\b', re.IGNORECASE),
    re.compile(r'\bNavigate\s+to\b', re.IGNORECASE),
    re.compile(r'\bExpand\s+(your|the)\s+\*\*', re.IGNORECASE),       # "Expand the **sdp_1_bronze** schema"
    # Icon references (images in markdown = UI screenshot)
    re.compile(r'!\[.*?Icon.*?\]\('),                                 # ![Catalog Icon](./path)
    re.compile(r'!\[.*?(icon|button|select|settings).*?\]\(', re.IGNORECASE),
    # Pipeline editor / Jobs & Pipelines UI
    re.compile(r'\bJobs\s*(&|and)\s*Pipelines\b', re.IGNORECASE),
    re.compile(r'\bLakeflow\s+(Pipelines?\s+)?Editor\b', re.IGNORECASE),
    re.compile(r'\bPipeline\s+(graph|details|settings)\b', re.IGNORECASE),
    re.compile(r'\bOpen\s+in\s+Editor\b', re.IGNORECASE),
    re.compile(r'\bRun\s+pipeline\b', re.IGNORECASE),
    re.compile(r'\bDry\s+Run\b', re.IGNORECASE),
    # Settings and configuration via UI
    re.compile(r'\bgear\s+icon\b', re.IGNORECASE),
    re.compile(r'\bellipsis\s+icon\b', re.IGNORECASE),
    re.compile(r'\bthree-dot\s+menu\b', re.IGNORECASE),
    re.compile(r'\bSelect\s+\*\*Create\*\*', re.IGNORECASE),
    re.compile(r'\bSelect\s+\*\*Settings\*\*', re.IGNORECASE),
    re.compile(r'\bAdd\s+configuration\b', re.IGNORECASE),
    re.compile(r'\bSelect\s+\*\*Save\*\*', re.IGNORECASE),
]

# Minimum number of distinct pattern matches in a cell to qualify as a UI step
MIN_PATTERN_MATCHES = 2


def extract_section_header(content: str) -> str:
    """Extract the markdown section header (##, ###) from a cell's content."""
    # Look for markdown headers
    match = re.search(r'^#{1,4}\s+(.+?)$', content, re.MULTILINE)
    if match:
        # Clean markdown formatting
        header = match.group(1).strip()
        header = re.sub(r'\*\*(.+?)\*\*', r'\1', header)  # Remove bold
        header = re.sub(r'!\[.*?\]\(.*?\)', '', header)    # Remove images
        return header.strip()
    return ""


def get_ui_step_summary(content: str) -> str:
    """Generate a short summary of what UI actions are described in the cell."""
    actions = []
    if re.search(r'\bSelect\s+(the\s+)?\*\*Catalog\*\*', content, re.IGNORECASE):
        actions.append("Navigate Catalog")
    if re.search(r'\bJobs\s*(&|and)\s*Pipelines\b', content, re.IGNORECASE):
        actions.append("Jobs & Pipelines")
    if re.search(r'\bRun\s+pipeline\b', content, re.IGNORECASE):
        actions.append("Run pipeline")
    if re.search(r'\bDry\s+Run\b', content, re.IGNORECASE):
        actions.append("Dry run")
    if re.search(r'\bOpen\s+in\s+Editor\b', content, re.IGNORECASE):
        actions.append("Open in Editor")
    if re.search(r'\bSchedule\b', content, re.IGNORECASE) and re.search(r'\bpipeline\b', content, re.IGNORECASE):
        actions.append("Schedule pipeline")
    if re.search(r'\bSettings\b', content) and re.search(r'\b(Select|gear|configure)\b', content, re.IGNORECASE):
        actions.append("Configure settings")
    if re.search(r'\bExpand\b', content, re.IGNORECASE) and re.search(r'\b(schema|catalog|volume|folder)\b', content, re.IGNORECASE):
        actions.append("Explore catalog/schema")
    if re.search(r'\bCreate\b.*\b(ETL|Pipeline)\b', content, re.IGNORECASE):
        actions.append("Create pipeline")
    if re.search(r'\bright-click\b', content, re.IGNORECASE):
        actions.append("Right-click menu")
    if re.search(r'\bPipeline\s+graph\b', content, re.IGNORECASE):
        actions.append("Pipeline graph")
    if re.search(r'\bevent.*log\b', content, re.IGNORECASE):
        actions.append("Event log")
    if re.search(r'\bEnable\b.*\b(Lakeflow|Editor|feature)\b', content, re.IGNORECASE):
        actions.append("Enable feature")
    return ", ".join(actions) if actions else "UI interaction"


def scan_notebook_for_ui_steps(notebook_path: str) -> list:
    """Scan a notebook for markdown cells containing UI instructions.
    
    Returns a list of dicts with section header and summary for each UI step found.
    """
    ui_steps = []
    
    try:
        export_resp = w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
        nb_json = json.loads(base64.b64decode(export_resp.content))
    except Exception as e:
        return [{"section": "ERROR", "summary": str(e)}]
    
    cells = nb_json.get("cells", [])
    
    for cell in cells:
        source = ''.join(cell.get("source", []))
        
        # Only check markdown cells and code cells with %md magic (used in Databricks)
        is_markdown = cell.get("cell_type") == "markdown"
        is_md_magic = cell.get("cell_type") == "code" and bool(re.match(r'\s*%md', source))
        
        if not (is_markdown or is_md_magic):
            continue
        
        # Count how many distinct UI patterns match in this cell
        matched_patterns = sum(1 for p in UI_ACTION_PATTERNS if p.search(source))
        
        if matched_patterns >= MIN_PATTERN_MATCHES:
            section = extract_section_header(source)
            summary = get_ui_step_summary(source)
            # Avoid duplicate entries for the same section
            if not any(s["section"] == section and section for s in ui_steps):
                ui_steps.append({
                    "section": section or "(unlabeled section)",
                    "summary": summary,
                    "pattern_matches": matched_patterns,
                })
    
    return ui_steps


# ── Scan All Course Notebooks ─────────────────────────────────────────────────
# Only scan Demos and Labs (lectures are presentation-only, no UI steps expected)

print("═" * 70)
print("AUTO-SCAN: NOTEBOOKS WITH UI INSTRUCTIONS")
print("═" * 70)
print("\nScanning all Demo and Lab notebooks for UI instruction patterns...\n")

UI_INSTRUCTION_NOTEBOOKS = {}
total_scanned = 0

for task in COURSE_TASKS:
    notebook_name = task["name"]
    # Scan Demos and Labs (skip Lectures and overview/summary notebooks)
    if not any(keyword in notebook_name for keyword in ["Demo", "Lab"]):
        continue
    
    notebook_path = os.path.normpath(f"{course_root}/{task['relative_path']}")
    total_scanned += 1
    
    ui_steps = scan_notebook_for_ui_steps(notebook_path)
    
    if ui_steps:
        UI_INSTRUCTION_NOTEBOOKS[notebook_name] = ui_steps

# ── Display Results ───────────────────────────────────────────────────────────
print(f"{len(UI_INSTRUCTION_NOTEBOOKS)} of {total_scanned} Demo/Lab notebooks contain UI steps:\n")

for notebook_name, steps in UI_INSTRUCTION_NOTEBOOKS.items():
    print(f"📋 {notebook_name}  ({len(steps)} UI sections)")
    for step in steps:
        section = step['section']
        summary = step['summary']
        print(f"     • {section} — {summary}")
    print()

total_steps = sum(len(s) for s in UI_INSTRUCTION_NOTEBOOKS.values())
print("═" * 70)
print(f"SCANNED: {total_scanned} notebooks  |  WITH UI STEPS: {len(UI_INSTRUCTION_NOTEBOOKS)}  |  TOTAL UI SECTIONS: {total_steps}")
print("═" * 70)

In [0]:
# Catalog / schema / table names
RESULTS_CATALOG   = "dbacademy"
RESULTS_SCHEMA    = spark.sql("SELECT current_user()").collect()[0][0].split("@")[0].replace(".", "_").replace("-", "_")
RESULTS_TABLE     = "notebook_run_results"
QA_FINDINGS_TABLE = "qa_content_findings"

# How long to wait for the entire job run (seconds)
TOTAL_RUN_TIMEOUT_SECONDS = 60 * 60 * 2   # 2 hour ceiling
POLL_INTERVAL_SECONDS     = 15

# Job and dashboard names
JOB_NAME       = f"[Course Validation] {COURSE_NAME}"
DASHBOARD_NAME = f"[Course Validation] {COURSE_NAME} Results Dashboard"

# Build the task list from the reusable course configuration above.
TASKS = []

for task in COURSE_TASKS:
    TASKS.append({
        "task_key": task["task_key"],
        "course": COURSE_NAME,
        "name": task["name"],
        "relative_path": task["relative_path"],
        "depends_on": task.get("depends_on", []),
        "env_key": task["env_key"],
        "use_classic": task.get("use_classic", False),
    })

if RUN_QA_CHECKER:
    TASKS.append({
        "task_key": "qa_content_checker",
        "course": "QA",
        "name": QA_TASK_NAME,
        "relative_path": QA_TASK_RELATIVE_PATH,
        "depends_on": [],
        "env_key": "qa_env",
    })

## Imports and workspace client

In [0]:
import json
import os
import time
from datetime import datetime, timezone

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.compute import Environment
from databricks.sdk.service.jobs import (
    JobEmailNotifications,
    JobEnvironment,
    NotebookTask,
    RunLifeCycleState,
    RunResultState,
    Task,
    TaskDependency,
)

from pyspark.sql import Row
from pyspark.sql.types import (
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

w = WorkspaceClient()

## Resolve workspace paths

In [0]:
this_notebook_path = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
course_root = "/".join(this_notebook_path.split("/")[:-1])

for t in TASKS:
    # os.path.normpath resolves '../' so Databricks gets a clean absolute path
    t["notebook_path"] = os.path.normpath(f"{course_root}/{t['relative_path']}")

print(f"Course root: {course_root}\n")
for t in TASKS:
    print(f"  [{t['task_key']}]  {t['name']}\n      {t['notebook_path']}")

## Ensure results table exists

In [0]:
results_schema_name = RESULTS_SCHEMA if RESULTS_SCHEMA != "information_schema" else "default"
results_fqn = f"{RESULTS_CATALOG}.{results_schema_name}.{RESULTS_TABLE}"

results_schema = StructType([
    StructField("run_timestamp",    TimestampType(), nullable=False),
    StructField("job_id",           LongType(),      nullable=True),
    StructField("job_run_id",       LongType(),      nullable=True),
    StructField("course",           StringType(),    nullable=True),
    StructField("task_key",         StringType(),    nullable=False),
    StructField("demo_name",        StringType(),    nullable=False),
    StructField("notebook_path",    StringType(),    nullable=False),
    StructField("status",           StringType(),    nullable=False),  # PASS / FAIL / TIMEOUT
    StructField("result_state",     StringType(),    nullable=True),
    StructField("life_cycle_state", StringType(),    nullable=True),
    StructField("duration_seconds", DoubleType(),    nullable=True),
    StructField("run_id",           LongType(),      nullable=True),
    StructField("run_page_url",     StringType(),    nullable=True),
    StructField("error_message",    StringType(),    nullable=True),
])

try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {RESULTS_CATALOG}.{results_schema_name}")
except Exception as e:
    if "UNAUTHORIZED_ACCESS" in str(e) or "PERMISSION_DENIED" in str(e):
        # Fall back to user's own labuser catalog (derived from username)
        user_catalog = spark.sql("SELECT current_user()").collect()[0][0].split("@")[0].replace(".", "_").replace("-", "_")
        print(f"⚠️  No permission to create schema in '{RESULTS_CATALOG}'. Falling back to catalog: {user_catalog}")
        RESULTS_CATALOG = user_catalog
        results_fqn = f"{RESULTS_CATALOG}.{results_schema_name}.{RESULTS_TABLE}"
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {RESULTS_CATALOG}.{results_schema_name}")
    else:
        raise

if not spark.catalog.tableExists(results_fqn):
    spark.createDataFrame([], results_schema).write.format("delta").saveAsTable(results_fqn)
    print(f"Created results table: {results_fqn}")
else:
    print(f"Results table exists: {results_fqn}")

## Create (or re-create) the course Lakeflow Job

In [0]:
# Delete any existing job with the same name so we always get a clean definition.
ENVIRONMENT_VERSION = "5"

existing = [j for j in w.jobs.list(name=JOB_NAME)]
for j in existing:
    w.jobs.delete(job_id=j.job_id)
    print(f"Deleted existing job: {j.job_id} ({j.settings.name})")

# Build Task objects from TASKS config.
import re

def _safe_key(k):
    return re.sub(r'[^a-zA-Z0-9_-]', '_', k)

# All tasks share one environment (identical spec — consolidates to stay within the 10-env API limit).
SHARED_ENV_KEY = "shared_env"

job_tasks = []
for t in TASKS:
    job_tasks.append(
        Task(
            task_key=_safe_key(t["task_key"]),
            description=t["name"],
            notebook_task=NotebookTask(notebook_path=t["notebook_path"]),
            environment_key=SHARED_ENV_KEY,
            depends_on=[TaskDependency(task_key=_safe_key(dep['task_key'])) for dep in t["depends_on"]],
        )
    )

# ── Route classic-compute tasks to the existing cluster ────────────────────────
# Dynamically fetch the user's classic cluster
current_user = spark.sql("SELECT current_user()").collect()[0][0]
user_prefix = current_user.split("@")[0]

CLASSIC_CLUSTER_ID = None
for c in w.clusters.list():
    if c.creator_user_name == current_user or c.cluster_name == user_prefix:
        CLASSIC_CLUSTER_ID = c.cluster_id
        print(f"Found classic cluster: {c.cluster_name} (ID: {c.cluster_id})")
        break

if not CLASSIC_CLUSTER_ID:
    raise RuntimeError("❌ No classic cluster found for user. All tasks require labuser compute — cannot proceed with serverless.")

if CLASSIC_CLUSTER_ID:
    for i, t in enumerate(TASKS):
        job_tasks[i].environment_key = None
        job_tasks[i].existing_cluster_id = CLASSIC_CLUSTER_ID

created_job = w.jobs.create(
    name=JOB_NAME,
    tasks=job_tasks,
    email_notifications=JobEmailNotifications(
        on_failure=TESTER_EMAILS,
        on_success=TESTER_EMAILS,
    ),
)
job_id = created_job.job_id
print(f"Created Lakeflow Job: {job_id}  ({JOB_NAME})")
print(f"\nTask DAG:")
for t in TASKS:
    deps = " → depends on: " + ", ".join(dep["task_key"] for dep in t["depends_on"]) if t["depends_on"] else " (starts immediately)"
    compute_label = "[CLASSIC]" if t.get("use_classic") else "[SERVERLESS]"
    print(f"  {compute_label} {t['task_key']}{deps}")

## Validate Demo 5 - Deploying a Simple DAB

Verifies that the Demo 5 DAB folder has the expected structure:
- `databricks.yml` (bundle config)
- `src/create_bronze_table` and `src/create_silver_table` (job source notebooks)
- `Demo - Deploying a Simple DAB` (the demo notebook)

Demo 5 requires **classic compute** for `databricks bundle` CLI commands.

In [0]:
# ── Validate Demo 5 - Deploying a Simple DAB ────────────────────────────────────
# Demo 5 uses `databricks bundle` CLI commands to validate, deploy, run, and
# destroy a simple DAB. The notebook itself handles job creation via DAJobConfig.
# This cell validates that the expected folder structure exists.
# ─────────────────────────────────────────────────────────────────────────────────

import os

# Derive the Demo 5 folder path
demo5_folder = os.path.normpath(f"{course_root}/../../05 Demo - Deploying a Simple DAB")
demo5_workspace_path = "/Workspace" + demo5_folder

print("═" * 70)
print("VALIDATE: Demo 5 - Deploying a Simple DAB")
print("═" * 70)
print(f"\nDemo 5 folder: {demo5_workspace_path}")

# Expected files/notebooks in the Demo 5 folder
expected_items = [
    ("Demo - Deploying a Simple DAB", "notebook"),
    ("databricks.yml", "file"),
    ("src", "folder"),
]

# Validate folder contents
all_ok = True
try:
    folder_contents = w.workspace.list(demo5_folder)
    found_names = {obj.path.split("/")[-1] for obj in folder_contents}
    
    for item_name, item_type in expected_items:
        if item_name in found_names:
            print(f"  ✅ {item_type}: {item_name}")
        else:
            print(f"  ❌ MISSING {item_type}: {item_name}")
            all_ok = False

    # Validate src/ has the expected notebooks
    src_path = f"{demo5_folder}/src"
    src_contents = w.workspace.list(src_path)
    src_names = {obj.path.split("/")[-1] for obj in src_contents}
    
    for nb in ["create_bronze_table", "create_silver_table"]:
        if nb in src_names:
            print(f"  ✅ src notebook: {nb}")
        else:
            print(f"  ❌ MISSING src notebook: {nb}")
            all_ok = False

except Exception as e:
    print(f"  ❌ ERROR accessing Demo 5 folder: {e}")
    all_ok = False

print()
if all_ok:
    print("✅ Demo 5 folder structure validated successfully.")
    print("   Note: Demo 5 runs on CLASSIC compute (uses databricks bundle CLI).")
else:
    print("⚠️  Demo 5 folder has missing items — the demo may fail.")
print("═" * 70)

## Demo 5 - Patch databricks.yml Resources

Demo 5 (Deploying a Simple DAB) requires patching the `databricks.yml` file:
- The template has an empty `resources` section (student TODO)
- We write the correct job resource (`demo05_simple_dab`) so `databricks bundle run` succeeds
- Requires **classic compute** (handled via `use_classic: True` in COURSE_TASKS)

In [0]:
# ── Patch Demo 5: Write resources section into databricks.yml ────────────────
# Demo 5 (Deploying a Simple DAB) requires the student to manually paste
# their job YAML into databricks.yml. For automated testing, we write the
# correct resources section so `databricks bundle run` can find the resource.
# ─────────────────────────────────────────────────────────────────────────────────

import os
import base64
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat

w = WorkspaceClient()

# Get username for the job name
username = spark.sql("SELECT current_user()").collect()[0][0].split("@")[0].replace(".", "_").replace("-", "_")

# Path to the databricks.yml
demo5_folder = os.path.normpath(f"{course_root}/../../05 Demo - Deploying a Simple DAB")
yml_path = f"{demo5_folder}/databricks.yml"

# Write the complete databricks.yml with the resources section filled in
yml_content = f"""###########################################################################################
# THIS IS THE MAIN DATABRICKS ASSET BUNDLE CONFIGURATION FOR THE PROJECT                  
###########################################################################################
# See https://docs.databricks.com/dev-tools/bundles/index.html for documentation.         #
###########################################################################################


################################################################################
# Bundle name  
# - The bundle mapping is required and must include a bundle name.
################################################################################
bundle:                   # Required
  name: demo05_bundle     # Required


############################################################################
# RESOURCES
############################################################################
resources:
  jobs:
    demo05_simple_dab:
      name: demo05_simple_dab_{username}
      tasks:
        - task_key: create_bronze_table
          notebook_task:
            notebook_path: ./src/create_bronze_table.ipynb
        - task_key: create_silver_table
          depends_on:
            - task_key: create_bronze_table
          notebook_task:
            notebook_path: ./src/create_silver_table.ipynb
      parameters:
        - name: catalog_name
          default: {username}_1_dev
        - name: display_target
          default: development


##############################################################################################
# TARGET DEPLOYMENT ENVIRONMENTS
##############################################################################################
targets:

  development:
    mode: development
    default: true
    workspace:
      root_path: /Workspace/Users/${{workspace.current_user.userName}}/.bundle/${{bundle.name}}/${{bundle.target}}
"""

# Write the patched databricks.yml
w.workspace.import_(
    path=yml_path,
    content=base64.b64encode(yml_content.encode()).decode(),
    format=ImportFormat.AUTO,
    overwrite=True,
)

print(f"✅ Demo 5: Patched databricks.yml with resources section")
print(f"   Job key: demo05_simple_dab")
print(f"   Job name: demo05_simple_dab_{username}")
print(f"   Path: {yml_path}")

In [0]:
# ── Patch DAJobConfig: Make check_for_duplicate_job_name idempotent ──────────
# On job retries, the demo notebook's DAJobConfig call fails with:
#   AssertionError: You already have a job with the same name.
# Fix: Patch Classroom-Setup-Common-Python to DELETE the existing job
# instead of raising an error, making the notebook safe to re-run.
# ─────────────────────────────────────────────────────────────────────────────────

import base64
from databricks.sdk.service.workspace import ExportFormat, ImportFormat, Language

# Path to the Classroom-Setup-Common-Python notebook
common_setup_path = os.path.normpath(f"{course_root}/../../Includes/Classroom-Setup-Common-Python")

# Export the notebook content
exported = w.workspace.export(path=common_setup_path, format=ExportFormat.SOURCE)
notebook_content = base64.b64decode(exported.content).decode("utf-8")

# Replace the check_for_duplicate_job_name method
old_method = """    # Check if the job name already exists, return error if it does.
    def check_for_duplicate_job_name(self, check_job_name: str):
        for job in self.w.jobs.list():
            if job.settings.name == check_job_name:
                test_job_name = False
                assert test_job_name, f'You already have a job with the same name. Please manually delete the job {self.job_name}'"""

new_method = """    # Check if the job name already exists; delete it if found (idempotent for retries).
    def check_for_duplicate_job_name(self, check_job_name: str):
        for job in self.w.jobs.list():
            if job.settings.name == check_job_name:
                print(f'Found existing job with name {check_job_name} (ID: {job.job_id}). Deleting it for a clean re-run...')
                self.w.jobs.delete(job_id=job.job_id)
                print(f'Deleted job {job.job_id}. Proceeding with creation.')"""

if old_method in notebook_content:
    patched_content = notebook_content.replace(old_method, new_method)
    w.workspace.import_(
        path=common_setup_path,
        content=base64.b64encode(patched_content.encode()).decode(),
        format=ImportFormat.SOURCE,
        language=Language.PYTHON,
        overwrite=True,
    )
    print(f"✅ Patched DAJobConfig.check_for_duplicate_job_name to delete existing jobs")
    print(f"   Path: {common_setup_path}")
elif new_method in notebook_content:
    print(f"ℹ️  DAJobConfig already patched (idempotent check). No changes needed.")
else:
    print(f"⚠️  Could not find the expected check_for_duplicate_job_name method to patch.")
    print(f"   The method signature may have changed. Manual review needed.")

In [0]:
# ── Patch 06L: Strip solution markers from notebook + write databricks.yml ──
# Lab 06L (Deploy a Simple DAB) has code cells wrapped in HTML-like markers:
#   <!-------------------ADD SOLUTION CODE BELOW------------------->
#   <actual solution code>
#   <!-------------------END SOLUTION CODE------------------->
# These markers cause SyntaxError when the notebook runs. This cell:
#   1. Strips the markers, leaving only the solution code
#   2. Replaces the placeholder job key with the user-specific one
#   3. Writes the correct databricks.yml with the resources section
# ─────────────────────────────────────────────────────────────────────────────────

import os
import re
import base64
import json
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat, Language

w = WorkspaceClient()

# Get username for personalization
username = spark.sql("SELECT current_user()").collect()[0][0].split("@")[0].replace(".", "_").replace("-", "_")

# ── 1. Patch the 06L notebook to strip solution markers ──────────────────────
lab06_folder = os.path.normpath(f"{course_root}/../../06L - Deploy a Simple DAB")
lab06_nb_path = f"{lab06_folder}/Lab - Deploy a Simple DAB"

print("═" * 70)
print("PATCH: 06L - Deploy a Simple DAB")
print("═" * 70)
print(f"\nNotebook: {lab06_nb_path}")

# Export notebook as source
export_resp = w.workspace.export(path=lab06_nb_path, format=ExportFormat.SOURCE)
nb_content = base64.b64decode(export_resp.content).decode()

# Strip solution markers from the content
# Pattern: <!---...ADD SOLUTION CODE BELOW...--->  and  <!---...END SOLUTION CODE...--->
marker_begin = re.compile(r'^\s*<!-+\s*ADD SOLUTION CODE BELOW\s*-+>\s*$', re.MULTILINE)
marker_end = re.compile(r'^\s*<!-+\s*END SOLUTION CODE\s*-+>\s*$', re.MULTILINE)

# Remove the marker lines
nb_content_patched = marker_begin.sub('', nb_content)
nb_content_patched = marker_end.sub('', nb_content_patched)

# Replace placeholder job key with user-specific one
nb_content_patched = nb_content_patched.replace('lab06_job_labuser123', f'lab06_job_{username}')

# Re-import the patched notebook
w.workspace.import_(
    path=lab06_nb_path,
    content=base64.b64encode(nb_content_patched.encode()).decode(),
    format=ImportFormat.SOURCE,
    language=Language.PYTHON,
    overwrite=True,
)
print("  ✅ Stripped solution markers from code cells")
print(f"  ✅ Replaced job key placeholder with: lab06_job_{username}")

# ── 2. Write the 06L databricks.yml with resources section ────────────────────
yml_path = f"{lab06_folder}/databricks.yml"

# Get the cluster ID for the existing_cluster_id field
# Re-detect the all-purpose cluster fresh (filter for RUNNING state) to avoid
# using a stale or non-all-purpose cluster ID from an earlier cell.
current_user_email = spark.sql("SELECT current_user()").collect()[0][0]
_user_prefix = current_user_email.split("@")[0]
cluster_id = None
for _c in w.clusters.list():
    if (_c.creator_user_name == current_user_email or _c.cluster_name == _user_prefix) and str(_c.state) in ("State.RUNNING", "State.RESIZING"):
        cluster_id = _c.cluster_id
        break
if not cluster_id:
    cluster_id = CLASSIC_CLUSTER_ID  # fallback
print(f"  Using cluster ID for DAB: {cluster_id}")

yml_content = f"""###########################################################################################
# THIS IS THE MAIN DATABRICKS ASSET BUNDLE CONFIGURATION FOR THE PROJECT                  
###########################################################################################
# See https://docs.databricks.com/dev-tools/bundles/index.html for documentation.         #
###########################################################################################

################
# Bundle name  
################
bundle:                       # Required
  name: demo06_lab_bundle     # Required


############################################################################
# RESOURCES
############################################################################
resources:
  jobs:
    lab06_job_{username}:
      name: lab06_job_{username}
      tasks:
        - task_key: create_nyc_tables
          notebook_task:
            notebook_path: ./src/our_project_code.ipynb
            source: WORKSPACE
          existing_cluster_id: {cluster_id}
      queue:
        enabled: true
      parameters:
        - name: catalog_name
          default: {username}_1_dev
        - name: display_target
          default: Development


##############################################################################################
# TARGET DEPLOYMENT ENVIRONMENTS
##############################################################################################
targets:

  dev:
    mode: development
    default: true
    workspace:
      root_path: /Workspace/Users/${{workspace.current_user.userName}}/.bundle/${{bundle.name}}/${{bundle.target}}
"""

w.workspace.import_(
    path=yml_path,
    content=base64.b64encode(yml_content.encode()).decode(),
    format=ImportFormat.AUTO,
    overwrite=True,
)
print(f"  ✅ Wrote databricks.yml with resources section")
print(f"     Job key: lab06_job_{username}")
print(f"     Cluster ID: {cluster_id}")
print(f"     Path: {yml_path}")
print("═" * 70)

In [0]:
# ── Validate Demo 8 - Deploying a DAB to Multiple Environments ──────────────
# Demo 8 uses `databricks bundle` CLI commands to deploy a DAB with dev/prod
# targets. The notebook is self-contained and does NOT need pipeline patching.
# This cell validates that the expected folder structure exists.
# ─────────────────────────────────────────────────────────────────────────────────

import os

# Derive the Demo 8 folder path
demo8_folder = os.path.normpath(f"{course_root}/../../08 Demo - Deploying a DAB to Multiple Environments")
demo8_workspace_path = "/Workspace" + demo8_folder

print("═" * 70)
print("VALIDATE: Demo 8 - Deploying a DAB to Multiple Environments")
print("═" * 70)
print(f"\nDemo 8 folder: {demo8_workspace_path}")

# Expected files/notebooks in the Demo 8 folder
expected_items_08 = [
    ("Demo - Deploying a DAB to Multiple Environments", "notebook"),
    ("databricks.yml", "file"),
    ("src", "folder"),
    ("resources", "folder"),
]

# Validate folder contents
all_ok_08 = True
try:
    folder_contents_08 = w.workspace.list(demo8_folder)
    found_names_08 = {obj.path.split("/")[-1] for obj in folder_contents_08}
    
    for item_name, item_type in expected_items_08:
        if item_name in found_names_08:
            print(f"  ✅ {item_type}: {item_name}")
        else:
            print(f"  ❌ MISSING {item_type}: {item_name}")
            all_ok_08 = False

    # Validate src/ has the expected notebooks
    src_path_08 = f"{demo8_folder}/src"
    src_contents_08 = w.workspace.list(src_path_08)
    src_names_08 = {obj.path.split("/")[-1] for obj in src_contents_08}
    
    for nb in ["create_bronze_table", "create_silver_table"]:
        if nb in src_names_08:
            print(f"  ✅ src notebook: {nb}")
        else:
            print(f"  ❌ MISSING src notebook: {nb}")
            all_ok_08 = False

    # Validate resources/ has the job YAML
    res_path_08 = f"{demo8_folder}/resources"
    res_contents_08 = w.workspace.list(res_path_08)
    res_names_08 = {obj.path.split("/")[-1] for obj in res_contents_08}
    
    if "demo_08_job.job.yml" in res_names_08:
        print(f"  ✅ resource: demo_08_job.job.yml")
    else:
        print(f"  ❌ MISSING resource: demo_08_job.job.yml")
        all_ok_08 = False

except Exception as e:
    print(f"  ❌ ERROR accessing Demo 8 folder: {e}")
    all_ok_08 = False

print()
if all_ok_08:
    print("✅ Demo 8 folder structure validated successfully.")
    print("   Note: Demo 8 runs on CLASSIC compute (uses databricks bundle CLI).")
    print("   No patching needed — notebook is self-contained (DAB multi-env deployment).")
else:
    print("⚠️  Demo 8 folder has missing items — the demo may fail.")
print("═" * 70)

In [0]:
# ── Patch Demo 8: Fill in cluster lookup variable in databricks.yml ──────────
# Demo 8's databricks.yml has a placeholder '???????????????????' for the
# cluster lookup variable. Replace it with the user's actual cluster name
# (which matches the lab username) so `databricks bundle validate` succeeds.
# ─────────────────────────────────────────────────────────────────────────────────

import os
import base64
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat

w = WorkspaceClient()

# Get the user's cluster name (in labs, cluster name = username prefix)
username = spark.sql("SELECT current_user()").collect()[0][0].split("@")[0]

# Path to the Demo 8 databricks.yml
demo8_folder = os.path.normpath(f"{course_root}/../../08 Demo - Deploying a DAB to Multiple Environments")
yml_path = f"{demo8_folder}/databricks.yml"

# Export current content
exported = w.workspace.export(path=yml_path, format=ExportFormat.AUTO)
yml_content = base64.b64decode(exported.content).decode("utf-8")

# Replace the placeholder with the actual cluster name
PLACEHOLDER = "???????????????????"
if PLACEHOLDER in yml_content:
    patched_content = yml_content.replace(PLACEHOLDER, username)
    w.workspace.import_(
        path=yml_path,
        content=base64.b64encode(patched_content.encode()).decode(),
        format=ImportFormat.AUTO,
        overwrite=True,
    )
    print(f"✅ Demo 8: Patched databricks.yml — cluster lookup set to '{username}'")
    print(f"   Path: {yml_path}")
else:
    print(f"ℹ️  Demo 8: databricks.yml already has a cluster name (no placeholder found). No changes needed.")

In [0]:
# ── Validate 09L - Deploy a DAB to Multiple Environments ────────────────────
# Lab 09L uses `databricks bundle` CLI commands to deploy a DAB with dev/prod
# targets (similar to Demo 8 but as a hands-on lab).
# The notebook is self-contained and does NOT need pipeline patching.
# ─────────────────────────────────────────────────────────────────────────────────

import os

# Derive the 09L folder path
lab09_folder = os.path.normpath(f"{course_root}/../../09L - Deploy a DAB to Multiple Environments")
lab09_workspace_path = "/Workspace" + lab09_folder

print("═" * 70)
print("VALIDATE: 09L - Deploy a DAB to Multiple Environments")
print("═" * 70)
print(f"\n09L folder: {lab09_workspace_path}")

# Expected files/notebooks in the 09L folder
expected_items_09 = [
    ("Lab - Deploy a DAB to Multiple Environments", "notebook"),
    ("databricks.yml", "file"),
    ("src", "folder"),
    ("resources", "folder"),
    ("solution", "folder"),
]

# Validate folder contents
all_ok_09 = True
try:
    folder_contents_09 = w.workspace.list(lab09_folder)
    found_names_09 = {obj.path.split("/")[-1] for obj in folder_contents_09}
    
    for item_name, item_type in expected_items_09:
        if item_name in found_names_09:
            print(f"  ✅ {item_type}: {item_name}")
        else:
            print(f"  ❌ MISSING {item_type}: {item_name}")
            all_ok_09 = False

    # Validate src/ has the expected notebook
    src_path_09 = f"{lab09_folder}/src"
    src_contents_09 = w.workspace.list(src_path_09)
    src_names_09 = {obj.path.split("/")[-1] for obj in src_contents_09}
    
    if "our_project_code" in src_names_09:
        print(f"  ✅ src notebook: our_project_code")
    else:
        print(f"  ❌ MISSING src notebook: our_project_code")
        all_ok_09 = False

    # Validate resources/ has the job YAML
    res_path_09 = f"{lab09_folder}/resources"
    res_contents_09 = w.workspace.list(res_path_09)
    res_names_09 = {obj.path.split("/")[-1] for obj in res_contents_09}
    
    if "lab09_nyc.job.yml" in res_names_09:
        print(f"  ✅ resource: lab09_nyc.job.yml")
    else:
        print(f"  ❌ MISSING resource: lab09_nyc.job.yml")
        all_ok_09 = False

except Exception as e:
    print(f"  ❌ ERROR accessing 09L folder: {e}")
    all_ok_09 = False

print()
if all_ok_09:
    print("✅ 09L folder structure validated successfully.")
    print("   Note: 09L runs on CLASSIC compute (uses databricks bundle CLI).")
    print("   No patching needed — lab auto-fill handles FILL_IN placeholders.")
else:
    print("⚠️  09L folder has missing items — the lab may fail.")
print("═" * 70)

In [0]:
# ── Patch 09L: Strip solution markers + write complete databricks.yml ────────
# Lab 09L has code cells wrapped in HTML-like markers:
#   <!-------------------ADD SOLUTION CODE BELOW------------------->
#   <actual solution code>
#   <!-------------------END SOLUTION CODE------------------->
# These markers cause SyntaxError when the notebook runs. This cell:
#   1. Strips the markers from the lab notebook
#   2. Writes the completed databricks.yml (fills in TODOs)
#   3. Writes the completed resource YAML
# ─────────────────────────────────────────────────────────────────────────────────

import os
import re
import base64
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat, Language

w = WorkspaceClient()

username = spark.sql("SELECT current_user()").collect()[0][0].split("@")[0].replace(".", "_").replace("-", "_")

# ── 1. Patch the 09L notebook to strip solution markers ──────────────────────
lab09_folder = os.path.normpath(f"{course_root}/../../09L - Deploy a DAB to Multiple Environments")
lab09_nb_path = f"{lab09_folder}/Lab - Deploy a DAB to Multiple Environments"

print("═" * 70)
print("PATCH: 09L - Deploy a DAB to Multiple Environments")
print("═" * 70)
print(f"\nNotebook: {lab09_nb_path}")

export_resp = w.workspace.export(path=lab09_nb_path, format=ExportFormat.SOURCE)
nb_content = base64.b64decode(export_resp.content).decode()

marker_begin = re.compile(r'^\s*<!-+\s*ADD SOLUTION CODE BELOW\s*-+>\s*$', re.MULTILINE)
marker_end = re.compile(r'^\s*<!-+\s*END SOLUTION CODE\s*-+>\s*$', re.MULTILINE)

nb_content_patched = marker_begin.sub('', nb_content)
nb_content_patched = marker_end.sub('', nb_content_patched)

w.workspace.import_(
    path=lab09_nb_path,
    content=base64.b64encode(nb_content_patched.encode()).decode(),
    format=ImportFormat.SOURCE,
    language=Language.PYTHON,
    overwrite=True,
)
print("  ✅ Stripped solution markers from code cells")

# ── 2. Detect running cluster for existing_cluster_id ────────────────────────
current_user_email = spark.sql("SELECT current_user()").collect()[0][0]
_user_prefix = current_user_email.split("@")[0]
cluster_id = None
for _c in w.clusters.list():
    if (_c.creator_user_name == current_user_email or _c.cluster_name == _user_prefix) and str(_c.state) in ("State.RUNNING", "State.RESIZING"):
        cluster_id = _c.cluster_id
        break
if not cluster_id:
    cluster_id = CLASSIC_CLUSTER_ID
print(f"  Using cluster ID: {cluster_id}")

# ── 3. Write the completed databricks.yml ─────────────────────────────────────
yml_path = f"{lab09_folder}/databricks.yml"

yml_content = f"""###########################################################################################
# THIS IS THE MAIN DATABRICKS ASSET BUNDLE CONFIGURATION FOR THE PROJECT                  
###########################################################################################

bundle:
  name: demo09_lab_bundle

include:
  - ./resources/lab09_nyc.job.yml

variables:
  user_name:
    description: Lab user name
    default: {username}

  catalog_dev:
    description: Development catalog for the project.
    default: ${{var.user_name}}_1_dev

  catalog_prod:
    description: Production catalog for the project.
    default: ${{var.user_name}}_3_prod

targets:
  dev:
    mode: development
    default: true
    workspace:
      root_path: /Workspace/Users/${{workspace.current_user.userName}}/.bundle/${{bundle.name}}/${{bundle.target}}
    resources:
      jobs:
        lab09_dab:
          parameters:
            - name: catalog_name
              default: ${{var.catalog_dev}}

  prod:
    mode: production
    workspace:
      root_path: /Workspace/Users/${{workspace.current_user.userName}}/.bundle/${{bundle.name}}/${{bundle.target}}
    resources:
      jobs:
        lab09_dab:
          parameters:
            - name: catalog_name
              default: ${{var.catalog_prod}}
"""

w.workspace.import_(
    path=yml_path,
    content=base64.b64encode(yml_content.encode()).decode(),
    format=ImportFormat.AUTO,
    overwrite=True,
)
print(f"  ✅ Wrote completed databricks.yml")

# ── 4. Write the completed resource YAML ──────────────────────────────────────
resource_path = f"{lab09_folder}/resources/lab09_nyc.job.yml"

resource_content = f"""resources:
  jobs:
    lab09_dab:
      name: lab09_dab_${{workspace.current_user.userName}}
      tasks:
        - task_key: create_nyc_tables
          notebook_task:
            notebook_path: ../src/our_project_code.ipynb
            source: WORKSPACE
          existing_cluster_id: {cluster_id}
      queue:
        enabled: true
      parameters:
        - name: display_target
          default: ${{bundle.target}}
"""

w.workspace.import_(
    path=resource_path,
    content=base64.b64encode(resource_content.encode()).decode(),
    format=ImportFormat.AUTO,
    overwrite=True,
)
print(f"  ✅ Wrote resource YAML: lab09_nyc.job.yml")
print(f"     Job key: lab09_dab")
print("═" * 70)

In [0]:
# ── Validate Demo 13 - CI/CD with DABs ────────────────────────────────────────
# Demo 13 uses `databricks bundle` CLI commands to deploy a multi-file bundle
# with unit tests, an SDP pipeline, and a visualization notebook across
# dev/stage/prod targets. The notebook is self-contained.
# ─────────────────────────────────────────────────────────────────────────────────

import os

# Derive the Demo 13 folder path
demo13_folder = os.path.normpath(f"{course_root}/../../13 Demo - Continuous Integration and Continuous Deployment with DABs")
demo13_workspace_path = "/Workspace" + demo13_folder

print("═" * 70)
print("VALIDATE: Demo 13 - CI/CD with DABs")
print("═" * 70)
print(f"\nDemo 13 folder: {demo13_workspace_path}")

# Expected files/notebooks in the Demo 13 folder
expected_items_13 = [
    ("Demo - CICD with DABs", "notebook"),
    ("Full Project", "folder"),
]

# Validate folder contents
all_ok_13 = True
try:
    folder_contents_13 = w.workspace.list(demo13_folder)
    found_names_13 = {obj.path.split("/")[-1] for obj in folder_contents_13}
    
    for item_name, item_type in expected_items_13:
        if item_name in found_names_13:
            print(f"  ✅ {item_type}: {item_name}")
        else:
            print(f"  ❌ MISSING {item_type}: {item_name}")
            all_ok_13 = False

    # Validate Full Project/ has the expected structure
    fp_path = f"{demo13_folder}/Full Project"
    fp_contents = w.workspace.list(fp_path)
    fp_names = {obj.path.split("/")[-1] for obj in fp_contents}
    
    for item in ["databricks.yml", "pytest.ini", "run_unit_tests", "src", "resources", "tests"]:
        if item in fp_names:
            print(f"  ✅ Full Project/{item}")
        else:
            print(f"  ❌ MISSING Full Project/{item}")
            all_ok_13 = False

except Exception as e:
    print(f"  ❌ ERROR accessing Demo 13 folder: {e}")
    all_ok_13 = False

print()
if all_ok_13:
    print("✅ Demo 13 folder structure validated successfully.")
    print("   Note: Demo 13 runs on CLASSIC compute (uses databricks bundle CLI).")
    print("   No patching needed — notebook is self-contained (CI/CD multi-target deployment).")
else:
    print("⚠️  Demo 13 folder has missing items — the demo may fail.")
print("═" * 70)

In [0]:
# ── Demo 16 - Using VSCode with Databricks ───────────────────────────────────
# Demo 16 is a simple informational walkthrough (11 cells, mostly markdown):
#   - Prints the workspace URL
#   - Instructs how to generate a PAT for VS Code
#   - No code execution beyond spark.conf.get()
#
# No patching needed. Works on SERVERLESS or classic compute.
# ─────────────────────────────────────────────────────────────────────────────────

print("Demo 16 (Using VSCode with Databricks): No notebook patching required.")
print("  • Simple walkthrough notebook (workspace URL + PAT generation).")
print("  • Only 1 executable Python cell (prints workspace URL).")
print("  • Works on serverless compute — no classic cluster needed.")

In [0]:
# ── Patch 14L Bonus Lab: Copy Solution Files to TODO Folder ──────────────────
# The 14L lab requires students to edit variables.yml and job YAML manually.
# For automated testing, we copy the solution files into the TODO folder so
# the `databricks bundle` CLI commands in the lab notebook succeed.
# ─────────────────────────────────────────────────────────────────────────────────

import os
import base64
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat

w = WorkspaceClient()

# Paths
lab_14_root = os.path.normpath(
    f"{course_root}/../../14L Bonus - Adding ML to Engineering Workflows with DABs"
)
solution_root = f"{lab_14_root}/Solution - Lab DABs Workflow"
todo_root = f"{lab_14_root}/TODO - Lab DABs Workflow"

# Get user-specific values for variable substitution
username = spark.sql("SELECT current_user()").collect()[0][0].split("@")[0].replace(".", "").replace("+", "")

print(f"Lab 14L root: {lab_14_root}")
print(f"Solution: {solution_root}")
print(f"TODO: {todo_root}")
print(f"Username for variables: {username}")

# ── 1. Copy solution variables.yml ────────────────────────────────────────────
src_variables = f"{solution_root}/resources/variables.yml"
dst_variables = f"{todo_root}/resources/variables.yml"

print(f"\nCopying: {src_variables}")
print(f"     To: {dst_variables}")

export_resp = w.workspace.export(path=src_variables, format=ExportFormat.AUTO)
variables_content = base64.b64decode(export_resp.content).decode()

# Replace placeholder username with actual user
variables_content = variables_content.replace("labuser1234", username)
variables_content = variables_content.replace("peter@fakeemail.com", f"{username}@databricks.com")

w.workspace.import_(
    path=dst_variables,
    content=base64.b64encode(variables_content.encode()).decode(),
    format=ImportFormat.AUTO,
    overwrite=True,
)
print("  ✅ Copied and personalized variables.yml")

# ── 2. Copy solution job YAML (with ML task) ─────────────────────────────────
src_job = f"{solution_root}/resources/job/dabs_workflow_with_ml.job.yml"
dst_job = f"{todo_root}/resources/job/dabs_workflow_with_ml.job.yml"

print(f"\nCopying: {src_job}")
print(f"     To: {dst_job}")

export_resp = w.workspace.export(path=src_job, format=ExportFormat.AUTO)
job_content = base64.b64decode(export_resp.content).decode()

w.workspace.import_(
    path=dst_job,
    content=base64.b64encode(job_content.encode()).decode(),
    format=ImportFormat.AUTO,
    overwrite=True,
)
print("  ✅ Copied dabs_workflow_with_ml.job.yml (includes ML_test task)")

print(f"\n✅ 14L Bonus Lab TODO folder patched with solution files")

# ── 3. Strip solution markers from the 14L lab notebook ───────────────────────
lab14_nb_path = f"{lab_14_root}/Lab - Adding ML to Engineering Workflows with DABs"
print(f"\nStripping solution markers from: {lab14_nb_path}")

export_resp = w.workspace.export(path=lab14_nb_path, format=ExportFormat.SOURCE)
nb_content_14 = base64.b64decode(export_resp.content).decode()

import re as _re
marker_begin_14 = _re.compile(r'^\s*<!-+\s*ADD SOLUTION CODE BELOW\s*-+>\s*', _re.MULTILINE)
marker_end_14 = _re.compile(r'^\s*<!-+\s*END SOLUTION CODE\s*-+>\s*', _re.MULTILINE)

nb_content_14_patched = marker_begin_14.sub('', nb_content_14)
nb_content_14_patched = marker_end_14.sub('', nb_content_14_patched)

# ── 3b. Fix cell 29: Replace error text placeholder with correct %sh command ──
# Cell 29 contains leftover error text that gets interpreted as Python,
# causing SyntaxError. Replace it with the bundle run command.
error_placeholder = "Error: Task Health_ETL failed!\nError:\nPlease refer to the logs for this pipeline in the pipelines page.\n"
correct_sh_command = "%sh\n# MAGIC cd \"./TODO - Lab DABs Workflow\"\n# MAGIC databricks bundle run ml_health_etl_workflow -t development"
nb_content_14_patched = nb_content_14_patched.replace(error_placeholder, correct_sh_command)

from databricks.sdk.service.workspace import Language as _Lang
w.workspace.import_(
    path=lab14_nb_path,
    content=base64.b64encode(nb_content_14_patched.encode()).decode(),
    format=ImportFormat.SOURCE,
    language=_Lang.PYTHON,
    overwrite=True,
)
print("  ✅ Stripped solution markers from 14L lab notebook")

## Trigger the job run and poll until all tasks complete

In [0]:
# ── Cleanup tables from prior runs to prevent TABLE_OR_VIEW_ALREADY_EXISTS ────
# Multiple tasks (06L, 08 Demo, 09L) create nyctaxi_raw / nyctaxi_bronze /
# nyctaxi_silver in the dev catalog. If a previous run left them behind,
# subsequent bundle runs fail with TABLE_OR_VIEW_ALREADY_EXISTS.
# Also cleans up Demo 05 tables (health_bronze_demo_05, health_silver_demo_05).
# ─────────────────────────────────────────────────────────────────────────────────

_cleanup_user = spark.sql("SELECT current_user()").collect()[0][0].split("@")[0].replace(".", "_").replace("-", "_")
_dev_catalog = f"{_cleanup_user}_1_dev"
_prod_catalog = f"{_cleanup_user}_3_prod"

_tables_to_drop = [
    "nyctaxi_raw",
    "nyctaxi_bronze",
    "nyctaxi_silver",
    "health_bronze_demo_05",
    "health_silver_demo_05",
]

print("═" * 70)
print("CLEANUP: Drop tables from previous test runs")
print("═" * 70)

for _cat in [_dev_catalog, _prod_catalog]:
    for _tbl in _tables_to_drop:
        _fqn = f"`{_cat}`.`default`.`{_tbl}`"
        try:
            spark.sql(f"DROP TABLE IF EXISTS {_fqn}")
            print(f"  Dropped (if existed): {_fqn}")
        except Exception as e:
            # Catalog may not exist yet (prod), skip gracefully
            if "SCHEMA_NOT_FOUND" in str(e) or "does not exist" in str(e).lower():
                pass
            else:
                print(f"  ⚠️  Could not drop {_fqn}: {e}")

print("\n✅ Table cleanup complete")
print("═" * 70)

In [0]:
run_response  = w.jobs.run_now(job_id=job_id)
job_run_id    = run_response.run_id
run_timestamp = datetime.now(timezone.utc)

print(f"Triggered job run: {job_run_id}")
print(f"Polling every {POLL_INTERVAL_SECONDS}s (timeout={TOTAL_RUN_TIMEOUT_SECONDS}s)...\n")

deadline        = time.time() + TOTAL_RUN_TIMEOUT_SECONDS
final_run       = None
terminal_states = {RunLifeCycleState.TERMINATED, RunLifeCycleState.SKIPPED, RunLifeCycleState.INTERNAL_ERROR}

while time.time() < deadline:
    run = w.jobs.get_run(run_id=job_run_id)
    lc  = run.state.life_cycle_state if run.state else None

    task_states = {
        tk.task_key: (
            tk.state.life_cycle_state.value if tk.state and tk.state.life_cycle_state else "PENDING"
        )
        for tk in (run.tasks or [])
    }
    print(f"  [{datetime.now(timezone.utc).strftime('%H:%M:%S')}] run={lc}  tasks={task_states}")

    if lc in terminal_states:
        final_run = run
        break

    time.sleep(POLL_INTERVAL_SECONDS)
else:
    print("TIMEOUT — cancelling the run.")
    try:
        w.jobs.cancel_run(run_id=job_run_id)
    except Exception:
        pass
    final_run = w.jobs.get_run(run_id=job_run_id)

print(f"\nFinal run state: {final_run.state.life_cycle_state if final_run.state else 'UNKNOWN'}")

## Collect per-task results

In [0]:
task_cfg = {t["task_key"]: t for t in TASKS}

results = []
for task_run in (final_run.tasks or []):
    cfg   = task_cfg.get(task_run.task_key, {})
    state = task_run.state

    result_state = state.result_state.value     if state and state.result_state     else None
    life_cycle   = state.life_cycle_state.value  if state and state.life_cycle_state  else None
    error_msg    = state.state_message           if state                             else None

    duration = None
    if task_run.start_time and task_run.end_time:
        duration = float((task_run.end_time - task_run.start_time) / 1000.0)

    if life_cycle == RunLifeCycleState.TERMINATED.value and result_state == RunResultState.SUCCESS.value:
        status = "PASS"
    elif life_cycle in (RunLifeCycleState.SKIPPED.value, RunLifeCycleState.INTERNAL_ERROR.value):
        status = "FAIL"
    elif life_cycle == RunLifeCycleState.TERMINATED.value:
        status = "FAIL"
    else:
        status = "TIMEOUT"

    results.append({
        "job_id":           job_id,
        "job_run_id":       job_run_id,
        "course":           cfg.get("course", ""),
        "task_key":         task_run.task_key,
        "demo_name":        cfg.get("name", task_run.task_key),
        "notebook_path":    cfg.get("notebook_path", ""),
        "status":           status,
        "result_state":     result_state,
        "life_cycle_state": life_cycle,
        "duration_seconds": duration,
        "run_id":           task_run.run_id,
        "run_page_url":     task_run.run_page_url,
        "error_message":    error_msg,
    })

# Sort back into configured TASKS order
order = {t["task_key"]: i for i, t in enumerate(TASKS)}
results.sort(key=lambda r: order.get(r["task_key"], 999))

## Append results to Delta + display summary

In [0]:
rows       = [Row(run_timestamp=run_timestamp, **r) for r in results]
results_df = spark.createDataFrame(rows, schema=results_schema)

results_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(results_fqn)

print(f"Appended {results_df.count()} rows to {results_fqn}\n")

# Summary for the configured single-course run
summary_groups = [COURSE_NAME]
if RUN_QA_CHECKER:
    summary_groups.append("QA")

for group_name in summary_groups:
    group_results = [r for r in results if r["course"] == group_name]
    if not group_results:
        continue

    passed = sum(1 for r in group_results if r["status"] == "PASS")
    failed = sum(1 for r in group_results if r["status"] != "PASS")
    icon = "✅" if failed == 0 else "❌"
    label = "QA Checks" if group_name == "QA" else COURSE_NAME
    print(f"  {icon} {label}:  {passed}/{len(group_results)} passed")

print(f"\nOverall: {sum(1 for r in results if r['status'] == 'PASS')}/{len(results)} tasks passed")

display(results_df)

## Create Lakeview Dashboard

In [0]:
import json
from databricks.sdk.service.dashboards import Dashboard

# ── Dashboard Configuration ────────────────────────────────────────────────────
DASHBOARD_FOLDER = "/".join(this_notebook_path.split("/")[:-2])  # Up from CourseRunner/
qa_findings_fqn = f"{RESULTS_CATALOG}.{RESULTS_SCHEMA}.{QA_FINDINGS_TABLE}"
# The QA checker may write to the 'default' schema — check both locations
if not spark.catalog.tableExists(qa_findings_fqn):
    qa_findings_alt = f"{RESULTS_CATALOG}.default.{QA_FINDINGS_TABLE}"
    if spark.catalog.tableExists(qa_findings_alt):
        qa_findings_fqn = qa_findings_alt
course_label_sql = COURSE_NAME.replace("'", "''")

# ── Dataset SQL (always filters to the latest run) ────────────────────────────
run_filter = f"job_run_id = (SELECT MAX(job_run_id) FROM {results_fqn})"
qa_filter  = f"run_timestamp = (SELECT MAX(run_timestamp) FROM {qa_findings_fqn})"

status_summary_sql = (
    f"SELECT status, COUNT(*) AS task_count "
    f"FROM {results_fqn} WHERE {run_filter} "
    f"GROUP BY status ORDER BY status"
)

task_detail_sql = (
    f"SELECT CASE WHEN course = 'QA' THEN 'QA Checks' ELSE '{course_label_sql}' END AS run_group, "
    f"task_key, demo_name, status, ROUND(duration_seconds, 1) AS duration_seconds, "
    f"run_page_url, error_message "
    f"FROM {results_fqn} WHERE {run_filter} "
    f"ORDER BY CASE WHEN course = 'QA' THEN 0 ELSE 1 END, task_key"
)

qa_sev_sql = (
    f"SELECT severity, COUNT(*) AS issue_count "
    f"FROM {qa_findings_fqn} WHERE {qa_filter} "
    f"GROUP BY severity ORDER BY issue_count DESC"
) if spark.catalog.tableExists(qa_findings_fqn) else (
    "SELECT 'No Data' AS severity, 0 AS issue_count WHERE 1=0"
)

qa_detail_sql = (
    f"SELECT notebook_name, cell_index, issue_type, severity, "
    f"offending_text, suggested_fix, check_source "
    f"FROM {qa_findings_fqn} WHERE {qa_filter} "
    f"ORDER BY severity, notebook_name, cell_index"
) if spark.catalog.tableExists(qa_findings_fqn) else (
    "SELECT '' AS notebook_name, 0 AS cell_index, '' AS issue_type, "
    "'' AS severity, '' AS offending_text, '' AS suggested_fix, '' AS check_source WHERE 1=0"
)

datasets = [
    {
        "name": "ds_status_summary",
        "displayName": "Task Status Summary",
        "query": status_summary_sql,
    },
    {
        "name": "ds_task_detail",
        "displayName": "Task Results Detail",
        "query": task_detail_sql,
    },
]

pages = [
    {
        "name": "pg_runs",
        "displayName": f"{COURSE_NAME} Run Results",
        "layout": [
            {
                "widget": {
                    "name": "w_bar",
                    "title": "Task Status — Latest Run",
                    "description": "",
                    "queries": [{
                        "name": "main",
                        "query": {
                            "datasetName": "ds_status_summary",
                            "fields": [
                                {"name": "status", "expression": "`status`"},
                                {"name": "task_count", "expression": "`task_count`"}
                            ],
                            "disaggregated": True
                        }
                    }],
                    "spec": {
                        "version": 3,
                        "widgetType": "bar",
                        "encodings": {
                            "x": {"fieldName": "status", "scale": {"type": "categorical"}},
                            "y": {"fieldName": "task_count", "scale": {"type": "quantitative"}}
                        }
                    }
                },
                "position": {"x": 0, "y": 0, "width": 6, "height": 6}
            },
            {
                "widget": {
                    "name": "w_task_table",
                    "title": "Task Results — Latest Run",
                    "description": "",
                    "queries": [{
                        "name": "main",
                        "query": {
                            "datasetName": "ds_task_detail",
                            "fields": [
                                {"name": "run_group", "expression": "`run_group`"},
                                {"name": "task_key", "expression": "`task_key`"},
                                {"name": "demo_name", "expression": "`demo_name`"},
                                {"name": "status", "expression": "`status`"},
                                {"name": "duration_seconds", "expression": "`duration_seconds`"},
                                {"name": "run_page_url", "expression": "`run_page_url`"},
                                {"name": "error_message", "expression": "`error_message`"}
                            ],
                            "disaggregated": True
                        }
                    }],
                    "spec": {
                        "version": 2,
                        "widgetType": "table",
                        "encodings": {
                            "columns": [
                                {"fieldName": "run_group"},
                                {"fieldName": "task_key"},
                                {"fieldName": "demo_name"},
                                {"fieldName": "status"},
                                {"fieldName": "duration_seconds"},
                                {"fieldName": "run_page_url"},
                                {"fieldName": "error_message"}
                            ]
                        }
                    }
                },
                "position": {"x": 0, "y": 6, "width": 12, "height": 8}
            }
        ]
    }
]

if RUN_QA_CHECKER:
    datasets.extend([
        {
            "name": "ds_qa_severity",
            "displayName": "QA Issues by Severity",
            "query": qa_sev_sql,
        },
        {
            "name": "ds_qa_detail",
            "displayName": "QA Findings Detail",
            "query": qa_detail_sql,
        },
    ])

    pages.append({
        "name": "pg_qa",
        "displayName": "QA Findings",
        "layout": [
            {
                "widget": {
                    "name": "w_qa_bar",
                    "title": "QA Issues by Severity — Latest Run",
                    "description": "",
                    "queries": [{
                        "name": "main",
                        "query": {
                            "datasetName": "ds_qa_severity",
                            "fields": [
                                {"name": "severity", "expression": "`severity`"},
                                {"name": "issue_count", "expression": "`issue_count`"}
                            ],
                            "disaggregated": True
                        }
                    }],
                    "spec": {
                        "version": 3,
                        "widgetType": "bar",
                        "encodings": {
                            "x": {"fieldName": "severity", "scale": {"type": "categorical"}},
                            "y": {"fieldName": "issue_count", "scale": {"type": "quantitative"}}
                        }
                    }
                },
                "position": {"x": 0, "y": 0, "width": 6, "height": 6}
            },
            {
                "widget": {
                    "name": "w_qa_table",
                    "title": "QA Findings Detail — Latest Run",
                    "description": "",
                    "queries": [{
                        "name": "main",
                        "query": {
                            "datasetName": "ds_qa_detail",
                            "fields": [
                                {"name": "notebook_name", "expression": "`notebook_name`"},
                                {"name": "cell_index", "expression": "`cell_index`"},
                                {"name": "issue_type", "expression": "`issue_type`"},
                                {"name": "severity", "expression": "`severity`"},
                                {"name": "offending_text", "expression": "`offending_text`"},
                                {"name": "suggested_fix", "expression": "`suggested_fix`"},
                                {"name": "check_source", "expression": "`check_source`"}
                            ],
                            "disaggregated": True
                        }
                    }],
                    "spec": {
                        "version": 2,
                        "widgetType": "table",
                        "encodings": {
                            "columns": [
                                {"fieldName": "notebook_name"},
                                {"fieldName": "cell_index"},
                                {"fieldName": "issue_type"},
                                {"fieldName": "severity"},
                                {"fieldName": "offending_text"},
                                {"fieldName": "suggested_fix"},
                                {"fieldName": "check_source"}
                            ]
                        }
                    }
                },
                "position": {"x": 0, "y": 6, "width": 12, "height": 8}
            }
        ]
    })

spec = {
    "datasets": datasets,
    "pages": pages,
}

# ── Auto-discover a SQL warehouse for the dashboard ───────────────────────────
from databricks.sdk.service.sql import State as WarehouseState

warehouse_id = None
try:
    for wh in w.warehouses.list():
        if wh.state in (WarehouseState.RUNNING, WarehouseState.STOPPED):
            warehouse_id = wh.id
            if wh.state == WarehouseState.RUNNING:
                break  # Prefer a running warehouse
except Exception as e:
    print(f"⚠️  Could not list warehouses: {e}")

if warehouse_id:
    print(f"Using SQL warehouse: {warehouse_id}")
else:
    print("⚠️  No SQL warehouse found — dashboard will have no data until one is assigned.")

# ── Create / re-create the Lakeview Dashboard ─────────────────────────────────
try:
    for d in w.lakeview.list():
        if d.display_name == DASHBOARD_NAME:
            w.lakeview.trash(dashboard_id=d.dashboard_id)
            print(f"Replaced existing dashboard: {d.dashboard_id}")
            break
except Exception:
    pass  # No existing dashboard — proceed to create

# ── Create / re-create the Lakeview Dashboard ─────────────────────────────────
# Clean up: remove any existing dashboard (API-created or file-based)
try:
    for d in w.lakeview.list():
        if d.display_name == DASHBOARD_NAME:
            state = str(d.lifecycle_state).upper() if d.lifecycle_state else ""
            if "TRASH" not in state:
                w.lakeview.trash(dashboard_id=d.dashboard_id)
                print(f"Replaced existing dashboard: {d.dashboard_id}")
except Exception:
    pass

# Also remove any .lvdash.json file with the same name
try:
    w.workspace.delete(path=f"{DASHBOARD_FOLDER}/{DASHBOARD_NAME}.lvdash.json")
except Exception:
    pass

dashboard = w.lakeview.create(Dashboard(
    display_name=DASHBOARD_NAME,
    serialized_dashboard=json.dumps(spec),
    parent_path=DASHBOARD_FOLDER,
    warehouse_id=warehouse_id,
))
dashboard_id = dashboard.dashboard_id
w.lakeview.publish(dashboard_id=dashboard_id, warehouse_id=warehouse_id, embed_credentials=True)

workspace_host = spark.conf.get("spark.databricks.workspaceUrl")
dashboard_url  = f"https://{workspace_host}/dashboardsv3/{dashboard_id}"

print(f"✅  Lakeview Dashboard created & published!")
print(f"    Name : {DASHBOARD_NAME}")
print(f"    ID   : {dashboard_id}")
print(f"    URL  : {dashboard_url}")
print(f"\n    ⚠️  NOTE: Lakeview API limitation — widget visualizations require one-time")
print(f"    activation via the dashboard editor. Open the dashboard and re-save widgets.")

# ── Inline Visualization (always works, regardless of dashboard rendering) ────
print("\n" + "═" * 70)
print("INLINE RESULTS VISUALIZATION")
print("═" * 70)

# Status summary chart
status_df = spark.sql(status_summary_sql)
display(status_df)

# QA severity chart (if table exists)
if spark.catalog.tableExists(qa_findings_fqn):
    qa_df = spark.sql(qa_sev_sql)
    display(qa_df)

In [0]:
# Display summary of failed tasks (deduplicated) with detailed error description
failed_rows = []
for r in results:
    if r["status"] != "PASS":
        name = r["demo_name"]
        if not any(row["Task"] == name for row in failed_rows):
            # Fetch detailed error from run output
            error_desc = ""
            try:
                run_output = w.jobs.get_run_output(run_id=r["run_id"])
                error_desc = (run_output.error or "").strip()
                if not error_desc and run_output.error_trace:
                    error_desc = run_output.error_trace.strip().split("\n")[-1]
            except Exception:
                pass
            failed_rows.append({
                "Task": name,
                "Error": r["error_message"] or "Workload failed",
                "Error Description": error_desc or "See run output for details",
            })

if failed_rows:
    failed_df = spark.createDataFrame(failed_rows)
    display(failed_df.select("Task", "Error", "Error Description"))
else:
    print("All tasks passed — no failures to report.")

## Fail the notebook if any task failed

In [0]:
failed = [r for r in results if r["status"] != "PASS"]
passed = [r for r in results if r["status"] == "PASS"]

if failed:
    summary = "\n".join(
        f"  - [{r['task_key']}] {r['demo_name']}: {r['status']} ({r['result_state']}) — {r['run_page_url']}"
        for r in failed
    )
    print(f"⚠️  {len(failed)} of {len(results)} task(s) failed on Serverless v{ENVIRONMENT_VERSION}:\n{summary}")
else:
    print(f"✅ All {len(results)} tasks passed on Serverless v{ENVIRONMENT_VERSION}.")

dbutils.notebook.exit(json.dumps({
    "total":  len(results),
    "passed": len(passed),
    "failed": len(failed),
}))